In [14]:
import torch

if torch.cuda.is_available():
    print("GPU is available")
else:
    print("GPU is not available")

GPU is available


In [2]:
# !pip install diffusers accelerate
# !pip install torchmetrics[image]
# !pip install git+https://github.com/openai/CLIP.git
# !pip install stable_baselines3
# !pip install shimmy
# !pip install scikit-image
# !pip install gym
# !pip install transformers
# !pip install ujson

In [3]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
import numpy as np
import matplotlib.pyplot as plt

from pycocotools.coco import COCO
import skimage.io as io
from PIL import Image

import gym
from gym import spaces
from stable_baselines3 import DQN
from stable_baselines3.common.vec_env import DummyVecEnv

from diffusers import StableDiffusionPipeline
from diffusers import DPMSolverMultistepScheduler as DefaultDPMSolver
from diffusers.utils import make_image_grid
from diffusers import EDMDPMSolverMultistepScheduler

from functools import partial
import clip
import torchvision.transforms as transforms
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.functional.multimodal import clip_score

import time
import pickle
import gc
# import ray
# from  ray.rllib.algorithms.dqn import DQN
# from ray.tune.registry import register_env


In [4]:
# Function to load images  ## added sample_first parameter
def load_all_images(coco, coco_caps, n=1000, sample_first=True):
    img_ids = coco.getImgIds()
    if sample_first:
        img_ids_list = img_ids[:n]
    else:
        img_ids_list = img_ids[-n:]
    
    imgs = []
    prompts = []
    for img_id in img_ids_list:
        img = coco.loadImgs(img_id)[0]
        ann_ids = coco_caps.getAnnIds(imgIds=img['id'])
        prompts.append(max([ann['caption'] for ann in coco_caps.loadAnns(ann_ids)], key=len))
        I = io.imread(img['coco_url'])
        imgs.append(Image.fromarray(I).convert('RGB'))
        del I  # Delete image to free memory
    return imgs, prompts

# Function to get text embedding
def get_text_embedding(text):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model, preprocess = clip.load("ViT-B/32", device=device)
    text_tokenized = clip.tokenize([text]).to(device)
    with torch.no_grad():
        text_embedding = model.encode_text(text_tokenized)
    return text_embedding.cpu().numpy().flatten()


# Text-to-Image Model ## Stable Diffusion 1.5
# Add support for setting custom timesteps
class DPMSolverMultistepScheduler(DefaultDPMSolver):
    def set_timesteps(
        self, num_inference_steps=None, device=None,
        timesteps=None
    ):
        if timesteps is None:
            super().set_timesteps(num_inference_steps, device)
            return

        all_sigmas = np.array(((1 - self.alphas_cumprod) / self.alphas_cumprod) ** 0.5)
        self.sigmas = torch.from_numpy(all_sigmas[timesteps])
        self.timesteps = torch.tensor(timesteps[:-1]).to(device=device, dtype=torch.int64) # Ignore the last 0

        self.num_inference_steps = len(timesteps)

        self.model_outputs = [
            None,
        ] * self.config.solver_order
        self.lower_order_nums = 0

        # add an index counter for schedulers that allow duplicated timesteps
        self._step_index = None
        self._begin_index = None
        self.sigmas = self.sigmas.to("cpu")  # to avoid too much CPU/GPU communication

# Function for text to image generation
model_id = "runwayml/stable-diffusion-v1-5"
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    pipe = StableDiffusionPipeline.from_pretrained(model_id, dtype=torch.float16, variant="fp16").to(device) 
else:
    pipe = StableDiffusionPipeline.from_pretrained(model_id).to(device)
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
def text_to_image(sampling_schedule, prompt):
    seed = 10
    # num_steps = len(sampling_schedule)-1
    torch.manual_seed(seed)
    images = pipe(
        prompt, num_images_per_prompt=1,
        timesteps=sampling_schedule,
    ).images
    return images[0]

# RL Framework
class ScheduleAdjustmentEnv(gym.Env):
    def __init__(self, base_schedule, text_to_image, text_embeddings, real_imgs, prompts):
        super(ScheduleAdjustmentEnv, self).__init__()

        self.base_schedule = base_schedule
        self.text_to_image = text_to_image
        self.text_embeddings = text_embeddings
        self.real_imgs = real_imgs
        self.prompts = prompts

        self.delta_t_ratio = [0.01, 0.02, 0.05, 0.1] #[0.01,0.05,0.1,0.5] #TO BE ADJUSTED
        self.weights = {"fid":1,"clip":1,"step":1} #TO BE ADJUSTED
        self.cur_index = 0

        self.generate_all()

        self.max_sample = 10
        self.fid_score_fn = FrechetInceptionDistance(normalize=True)
        self.fid_baseline = self.calculate_fid()

        self.clip_score_fn = partial(clip_score, model_name_or_path="openai/clip-vit-base-patch32")

        # Action space: (index*ratio*increase/decrease)
        self.action_space = spaces.Discrete((len(base_schedule)-2)*len(self.delta_t_ratio)*2)

        # Observation space: sampling_schedule, text_embedding
        self.observation_space = spaces.Dict({
            "sampling_schedule": spaces.Box(low=min(self.base_schedule), high=max(self.base_schedule), shape=(len(base_schedule),), dtype=np.int32),
            "text_embedding": spaces.Box(low=-np.inf, high=np.inf, shape=text_embeddings[0].shape, dtype=np.float32),
        })

    def step(self, action):
        self.steps += 1
        index, op = action//(len(self.delta_t_ratio)*2), action%(len(self.delta_t_ratio)*2)

        # Ensure valid index (ignore t_min and t_max)
        index += 1

        if  op%2==0:
            max_delta_t = self.sampling_schedule[index-1] - self.sampling_schedule[index]
            self.sampling_schedule[index] += int(max_delta_t*self.delta_t_ratio[op//2])
        else:
            max_delta_t = self.sampling_schedule[index] - self.sampling_schedule[index+1]
            self.sampling_schedule[index] -= int(max_delta_t*self.delta_t_ratio[op//2])

        self.generate()
        img = self.gen_imgs[self.cur_index]  ## added to output generated image
        fid = self.calculate_fid()
        clip = self.calculate_clip()

        reward = self.weights["clip"]*clip/self.clip_baseline - self.weights["fid"]*fid/self.fid_baseline - self.weights["step"]*self.steps

        # Adjust delta_t_ratio, weights, max_step based on performance metrics
        #self.adjust_schedule_parameters(fid, clip, reward)

        return {"sampling_schedule":self.sampling_schedule,"text_embedding":self.text_embeddings[self.cur_index]}, reward, {"gen_img": img}, False, {}
        
    def adjust_schedule_parameters(self, fid, clip, reward):
        # Example adjustment logic for delta_t_ratio, weights, and max_step
        # Adjust delta_t_ratio based on FID and clip score
        if fid > self.fid_baseline:
            # Increase delta_t_ratio to make the sampling schedule finer
            self.delta_t_ratio = [min(x + 0.05, 1.0) for x in self.delta_t_ratio]
        else:
            # Decrease delta_t_ratio to make the schedule coarser
            self.delta_t_ratio = [max(x - 0.05, 0.01) for x in self.delta_t_ratio]

        # Adjust weights to prioritize certain metrics (e.g., prioritize FID or Clip score)
        if clip < 0.5:
            self.weights["clip"] *= 1.1  # Increase importance of clip if the score is low
        else:
            self.weights["clip"] *= 0.9  # Decrease importance of clip if the score is high

        if fid > 50:
            self.weights["fid"] *= 1.1  # Increase importance of FID if it's high
        else:
            self.weights["fid"] *= 0.9  # Decrease importance of FID if it's low
        print(f"Adjusted weights: {self.weights}")

    def reset(self):
        print("reset")
        self.sampling_schedule = self.base_schedule.copy()
        self.clip_baseline = self.calculate_clip()
        self.fid_baseline = self.calculate_fid(True)
        self.steps = 0
        return {"sampling_schedule":self.sampling_schedule,"text_embedding":self.text_embeddings[self.cur_index]}

    def render(self, mode="human"):
        print("Sampling Schedule:", self.sampling_schedule)

    def set_index(self, i):
        self.cur_index = i

    def generate(self):
        self.gen_imgs[self.cur_index] = self.text_to_image(self.sampling_schedule, self.prompts[self.cur_index])

    def generate_all(self):
        self.gen_imgs = []
        for prompt in self.prompts:
            self.gen_imgs.append(self.text_to_image(self.base_schedule, prompt))
        self.baseline_imgs = self.gen_imgs.copy()

    def calculate_fid(self, baseline = False):
        first_index = max(0,self.cur_index-self.max_sample+1)
        last_index = max(self.max_sample,self.cur_index+1)
        self.fid_score_fn.reset()
        self.fid_score_fn.update(self.preprocess_fid(self.real_imgs[first_index:last_index]), real=True)
        if baseline:
            self.fid_score_fn.update(self.preprocess_fid(self.baseline_imgs[first_index:last_index]), real=False)
        else:
            self.fid_score_fn.update(self.preprocess_fid(self.gen_imgs[first_index:last_index]), real=False)
        return self.fid_score_fn.compute()

    def preprocess_fid(self, images):
        transform = transforms.Compose([
            transforms.Resize((299, 299)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        return torch.cat([transform(img).unsqueeze(0) for img in images])

    def calculate_clip(self):
#         clip = self.clip_score_fn(torch.from_numpy(np.array([self.gen_imgs[self.cur_index]])).permute(0, 3, 1, 2), [self.prompts[self.cur_index]]).detach()
        images = np.array(self.gen_imgs[self.cur_index])  # Convert to NumPy array
        images_tensor = torch.from_numpy(images).permute(2, 0, 1).unsqueeze(0)  # Convert to tensor
        clip_score = clip_score_fn(images_tensor, [self.prompts[self.cur_index]]).detach().cpu().item()
        return round(float(clip_score), 4)


Keyword arguments {'dtype': torch.float16} are not expected by StableDiffusionPipeline and will be ignored.


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

In [5]:
# Functions for Clip Score and FID score
clip_score_fn = partial(clip_score, model_name_or_path="openai/clip-vit-base-patch32")
def calculate_clip_score(images, prompts):
#     clip_score = clip_score_fn(torch.from_numpy(np.array([images])).permute(0, 3, 1, 2), [prompts]).detach()
    images = np.array(images)  # Convert to NumPy array
    images_tensor = torch.from_numpy(images).permute(2, 0, 1).unsqueeze(0)  # Convert to tensor
    clip_score = clip_score_fn(images_tensor, [prompts]).detach().cpu().item()
    return round(float(clip_score), 4)

def clip_list(gen_images, prompts):
    return [calculate_clip_score(gen_images[i], prompts[i]) for i in range(len(prompts))]

def preprocess_images(images):
        transform = transforms.Compose([
            transforms.Resize((299, 299)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        return torch.cat([transform(img).unsqueeze(0).to(device, dtype=dtype) for img in images])

fid_score_fn = FrechetInceptionDistance(normalize=True).to(device)
def calc_fid_score(gen_images):
    generated_images_tensor = preprocess_images(gen_images)
    real_images_tensor = preprocess_images(real_imgs)
    fid_score_fn.update(real_images_tensor, real=True)
    fid_score_fn.update(generated_images_tensor, real=False)
    return fid_score_fn.compute().cpu().item()


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float16 if torch.cuda.is_available() else torch.float32 

In [7]:
# data_dir = "./coco"
# ann_url = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
# os.makedirs(data_dir, exist_ok=True)

# print("Downloading COCO annotations...")
# os.system(f"wget {ann_url} -P {data_dir} && unzip {data_dir}/annotations_trainval2017.zip -d {data_dir}")
# print("COCO dataset downloaded successfully!")

In [8]:
# Import Dataset
data_dir = "./coco"
ann_file='{}/annotations/instances_val2017.json'.format(data_dir)
coco=COCO(ann_file)
ann_file = '{}/annotations/captions_val2017.json'.format(data_dir)
coco_caps=COCO(ann_file)

Loading annotations into memory...
Done (t=0.41s)
Creating index...
index created!
Loading annotations into memory...
Done (t=0.02s)
Creating index...
index created!


In [9]:
# Load last 100 image for testing
real_imgs, prompts = load_all_images(coco, coco_caps, n=100, sample_first=False)

In [10]:
actual_data = {'real_img': real_imgs, 'prompt':prompts}
with open("actual_data.pkl", "wb") as f:
        pickle.dump(actual_data, f)
print("Exported actual data.")

Exported actual data.


# Original AYS Model

In [11]:
model_id = "runwayml/stable-diffusion-v1-5"
pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=dtype, variant="fp16" if device == "cuda" else None).to(device)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
seed = 10
torch.manual_seed(seed)

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

In [12]:
ays_schedule = np.array([999, 850, 736, 645, 545, 455, 343, 233, 124, 24, 0]) 
start_time = time.time()
ays_gen_images = pipe(
    prompts, num_images_per_prompt=1,
    timesteps=ays_schedule,
).images
end_time = time.time()

ays_time = end_time - start_time
ays_clip_list = clip_list(ays_gen_images,prompts)
ays_fid_score = calc_fid_score(ays_gen_images)

  0%|          | 0/10 [00:00<?, ?it/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.50, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [13]:
ays_data =  {"gen_img": ays_gen_images, "clip_list":ays_clip_list, "fid_score": ays_fid_score, "time": ays_time}
with open("ays_result.pkl", "wb") as f:
        pickle.dump(ays_data, f)
print("Exported AYS result.")

Exported AYS result.


In [14]:
# free memory
torch.cuda.empty_cache()
gc.collect()

40

# Uniform Time Schedule

In [15]:
uniform_schedule = np.linspace(999, 0, 10).astype(int)
start_time = time.time()
uniform_gen_images = pipe(
    prompts, num_images_per_prompt=1,
    timesteps=uniform_schedule,
).images
end_time = time.time()

uniform_time = end_time - start_time
uniform_clip_list = clip_list(uniform_gen_images,prompts)
uniform_fid_score = calc_fid_score(uniform_gen_images)

  0%|          | 0/9 [00:00<?, ?it/s]

In [16]:
uniform_data =  {"gen_img": uniform_gen_images, "clip_list":uniform_clip_list, "fid_score": uniform_fid_score, "time": uniform_time}
with open("uniform_result.pkl", "wb") as f:
        pickle.dump(uniform_data, f)
print("Exported Uniform Time result.")

Exported Uniform Time result.


In [17]:
# free memory
torch.cuda.empty_cache()
gc.collect()

104

# EDM schedule

In [18]:
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

start_time = time.time()
edm_gen_images = pipe(
    prompts, num_images_per_prompt=1
).images
end_time = time.time()

edm_time = end_time - start_time
edm_clip_list = clip_list(edm_gen_images,prompts)
edm_fid_score = calc_fid_score(edm_gen_images)

  0%|          | 0/50 [00:00<?, ?it/s]

In [19]:
edm_data =  {"gen_img": edm_gen_images, "clip_list":edm_clip_list, "fid_score": edm_fid_score, "time": edm_time}
with open("edm_result.pkl", "wb") as f:
        pickle.dump(edm_data, f)
print("Exported EDM result.")

Exported EDM result.


In [20]:
# free memory
torch.cuda.empty_cache()
gc.collect()

48

# AYS-RL Model

In [10]:
model_id = "runwayml/stable-diffusion-v1-5"
if device == "cuda":
    pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=dtype, variant="fp16").to(device) 
else:
    pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=dtype).to(device)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
seed = 10
torch.manual_seed(seed)

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

In [11]:
# Set the environment
text_embeddings = [get_text_embedding(prompt) for prompt in prompts]
sampling_schedule = np.array([999, 850, 736, 645, 545, 455, 343, 233, 124, 24, 0])  # original AYS sampling schedule
env = ScheduleAdjustmentEnv(sampling_schedule, text_to_image, text_embeddings, real_imgs, prompts)
# Load trained AYS-RL model (trained by 2000 images with 20 iterations)
AYS_RL_model = DQN.load("./rl_model_2k.zip", env, verbose=1)   ##default guidance scale is 7.5

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

...

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/home/svu/e1352422/vanda_pypkg/pytorch_2.4.0a0-cuda_12.5.0_ngc_24.06/local/lib/python3.10/dist-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


In [15]:
aysrl_schedule = []
aysrl_gen_images = []
start_time = time.time()
for i in range(len(prompts)):
    print(f"index: {i}")  
    env.set_index(i)
    obs = env.reset()
    # last_reward = -np.inf
    for _ in range(10):
        action, _ = AYS_RL_model.predict(obs)
        obs, reward, img,  done, _ = env.step(action)
        env.render()

    #   aysrl_schedule.append(last_obs['sampling_schedule'])
    aysrl_schedule.append(obs['sampling_schedule'])
    aysrl_gen_images.append(img['gen_img'])
    # aysrl_gen_images.append(AYS_RL_model.generate())
end_time = time.time()

aysrl_time = end_time - start_time
aysrl_clip_list = clip_list(aysrl_gen_images,prompts)
aysrl_fid_score = calc_fid_score(aysrl_gen_images)

index: 0
reset


  0%|          | 0/10 [00:00<?, ?it/s]

Sampling Schedule: [999 850 736 645 555 455 343 233 124  24   0]


  0%|          | 0/10 [00:00<?, ?it/s]

Sampling Schedule: [999 850 736 645 564 455 343 233 124  24   0]


...

Sampling Schedule: [999 850 738 645 596 455 343 233 124  24   0]


  0%|          | 0/10 [00:00<?, ?it/s]

Sampling Schedule: [999 850 738 645 600 455 343 233 124  24   0]


  0%|          | 0/10 [00:00<?, ?it/s]

Sampling Schedule: [999 850 738 645 604 455 343 233 124  24   0]


In [16]:
aysrl_data =  {"gen_img": aysrl_gen_images, "clip_list":aysrl_clip_list, "fid_score": aysrl_fid_score, "time": aysrl_time}
with open("aysrl_2k_result.pkl", "wb") as f:
        pickle.dump(aysrl_data, f)
print("Exported AYS-RL result.")

Exported AYS-RL result.
